# 03 · Entraînements complets sur GPU (Kaggle) avec DVC + MLflow (DagsHub)

Ce notebook exécute **le même pipeline** que le projet (`dvc.yaml`) sur un GPU Kaggle, et journalise tout dans le **MLflow hébergé par DagsHub** (runs, métriques, artefacts, Model Registry). L'enseignante voit ainsi les vraies expériences sur l'interface MLflow.

**Prérequis Kaggle** (panneau de droite) :
1. *Accelerator* : **GPU T4 ×2** ou **P100** ; *Internet* : **On**.
2. *Add-ons → Secrets* : ajouter `DAGSHUB_TOKEN` (jeton DagsHub) et, si le dépôt GitHub est privé, `GITHUB_TOKEN`.
3. *(Optionnel, plus rapide que `dvc pull`)* : *Add Input*, puis le dataset `abdallahalidev/plantvillage-dataset`.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, subprocess
secrets = UserSecretsClient()
DAGSHUB_USER = "Nawres2000"
DAGSHUB_TOKEN = secrets.get_secret("DAGSHUB_TOKEN")
try:
    GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None
REPO = "github.com/Nawres2000/phytora.git"
url = f"https://{GITHUB_TOKEN}@{REPO}" if GITHUB_TOKEN else f"https://{REPO}"
subprocess.run(["git", "clone", "--depth", "50", url, "/kaggle/working/phytora"], check=True)
os.chdir("/kaggle/working/phytora")
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 1. Dépendances (PyTorch CUDA est déjà installé sur Kaggle)

In [ ]:
!grep -vE '^(torch|torchvision|--extra-index-url)' requirements-train.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (activez le GPU)")

## 2. Données : dataset Kaggle en entrée (rapide) ou `dvc pull` depuis DagsHub

In [ ]:
from pathlib import Path
import shutil
!dvc remote modify origin --local access_key_id {DAGSHUB_TOKEN}
!dvc remote modify origin --local secret_access_key {DAGSHUB_TOKEN}
kaggle_color = next(iter(Path("/kaggle/input").rglob("plantvillage-dataset/**/color")), None) if Path("/kaggle/input").exists() else None
dst = Path("data/raw/plantvillage/color")
if kaggle_color and not dst.exists():
    # copy (not symlink) + dvc add: the data must be in the DVC cache for `dvc exp run --temp`
    shutil.copytree(kaggle_color, dst)
    !dvc add data/raw/plantvillage
    print("dataset Kaggle copié et versionné :", kaggle_color)
elif not dst.exists():
    !dvc pull data/raw/plantvillage.dvc
!ls data/raw/plantvillage/color | wc -l

## 3. MLflow hébergé par DagsHub

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/phytora.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["PYTHONWARNINGS"] = "ignore"
print("MLflow :", os.environ["MLFLOW_TRACKING_URI"])

## 4. Grille d'expériences sur le **dataset complet** (`quick_n=0`)
Chaque configuration est un `dvc exp run --temp`, qui crée un run MLflow et une nouvelle version dans le Model Registry. La version est promue en production si elle bat la version en place. Durée indicative sur T4 : 15 à 40 min par configuration.

In [ ]:
from src.pipelines.experiments import experiment_grid, DEFAULT_GRID
print(DEFAULT_GRID)
summary = experiment_grid(quick_n=0)
summary["succeeded"], summary["failed"]

In [ ]:
print(open("reports/experiments.md").read())

## 5. Validation croisée (3 folds, sous-échantillon pour limiter le temps)

In [ ]:
from src.pipelines.experiments import cross_validation
cv = cross_validation(folds=3, quick_n=300, model="mobilenet_v3_small")
print(f"F1 macro CV = {cv['cv_macro_f1_mean']:.4f} ± {cv['cv_macro_f1_std']:.4f}")

## 6. Entraîner le modèle de production (pipeline principal) et pousser les artefacts

In [ ]:
!dvc repro
!dvc metrics show
!dvc push

## 7. Récupérer le résultat dans le dépôt
Téléchargez `dvc.lock` et `reports/*.json` (onglet *Output*), commitez-les dans le dépôt, puis `dvc pull` en local récupère le nouveau modèle. Avec un `GITHUB_TOKEN` en écriture, vous pouvez aussi pousser directement :

In [ ]:
if GITHUB_TOKEN:
    !git config user.name "{DAGSHUB_USER}" && git config user.email "145472309+Nawres2000@users.noreply.github.com"
    !git checkout -b kaggle-training && git add dvc.lock reports/*.json reports/*.md && git commit -m "Train on Kaggle GPU (full dataset)" && git push -u origin kaggle-training
    print("Ouvrez une Pull Request kaggle-training -> main sur GitHub")